In [120]:
import importlib as il
import numpy as np
import more_itertools as mit
import scipy.linalg as spl
import cvxpy as cp
import dikin_utils as du


In [14]:
def whiten(V: np.ndarray, eps = 1e-6):
    # M = (cov(V))^-1/2
    M = np.cov(V) + eps * np.eye(V.shape[0])
    M = np.linalg.inv(M)
    # M = np.linalg.cholesky(M)
    M = spl.sqrtm(M)
    return M

def gram_xform(V: np.ndarray, eps = 1e-6):
    # M = (V^2 + eps * I)^-1/2
    VV = V.T@V
    M = (VV + eps * np.eye(VV.shape[0]))
    M = np.linalg.inv(M)
    M = spl.sqrtm(M)
    return M

def exp_scale(V: np.ndarray, u: np.ndarray, eps = 1):
    # M = exp(e(u@uT)) where u is direction vector
    M = np.exp(eps * u@u.T)
    return M

def exp_scale_2(V: np.ndarray, u: np.ndarray, eps = 1):
    # M = e^s *V + (1-e^s)(uu^T)V
    M = np.exp(-eps) * V + (1 - np.exp(-eps)) * (u@u.T)@V
    return M

V = np.array([[-4, 1], [1, -4]])
M = whiten(V)
print("White:", M)
M = gram_xform(V)
print("Gram:", M)
M = exp_scale(V, np.array([-1, -1]).reshape(-1, 1))
print("Exp:", M)
M = exp_scale_2(V, np.array([-1, -1]).reshape(-1, 1), eps=1.5)
print("Exp2:", M)

White: [[500.09999996 499.89999996]
 [499.89999996 500.09999996]]
Gram: [[0.26666666 0.06666666]
 [0.06666666 0.26666666]]
Exp: [[2.71828183 2.71828183]
 [2.71828183 2.71828183]]
Exp2: [[-3.22313016 -2.10747936]
 [-2.10747936 -3.22313016]]


In [202]:
# this is where we experiment with different ways of acquiring a det(A) = 1 matrix:
# 1. try the inv_prod in CVX; why would this be better than -1/dim?
# 2. try objective ||N-M|| with N in Z; see if CVX can send it to gurobi
# 3. try the LLL

R = np.random.rand(20, 20) * 5
# normalize the columns:
R /= np.linalg.norm(R, axis=0)

def to_det_1(A: np.ndarray):
    # A = A * (det(A))^-1/20
    dim = A.shape[0]
    return A * (np.abs(np.linalg.det(A)) ** (-1/dim))

def to_int_1(A: np.ndarray):
    B = cp.Variable(A.shape, integer=True)
    # constraints = [cp.log_det(B) == 0]  # Not DCP. Darn.
    constraints = [cp.trace(B) == 1 if np.linalg.det(A) > 0 else cp.trace(B) == -1]
    problem = cp.Problem(cp.Minimize(cp.norm(B - A)), constraints)  # needs Mosek. Lame.
    problem.solve(verbose=True)
    return B.value

Rd = to_det_1(R)
#np.linalg.det(R), np.linalg.det(Rd), Rd
# Rn = np.round(Rd)
# np.linalg.det(Rn)

# t2 = to_int_1(Rd)
# np.linalg.det(t2)
Rint = np.round(Rd * 1000)
U = du.CLLL(Rint)
Rd2 = np.round(Rint / 1000)
np.linalg.det(Rint), np.linalg.det(U), np.linalg.det(Rd2)


(9.865525404952184e+59, -0.9999999999999999, 0.0)